In [ ]:
import pandas as pd 
import numpy as np 

In [ ]:
movies=pd.read_csv('tmdb_5000_movies.csv.zip')
credits=pd.read_csv('tmdb_5000_credits.csv.zip')

In [ ]:
movies.head(1)

In [ ]:
credits.head(1)

In [ ]:
movies=movies.merge(credits,on='title')

In [ ]:
movies.head(1)

In [ ]:
# genres 
# id
# keywords
# title
# overview
# cast
# crew
movies=movies[['genres','id','keywords','title','overview','cast','crew']]

In [ ]:
movies.head(1)

In [ ]:
movies.isnull().sum()

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies.duplicated().sum()

In [ ]:
movies.iloc[0].genres
import ast

In [ ]:
def convert(obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [ ]:
movies['genres']=movies['genres'].apply(convert)

In [ ]:
movies.head(1)

In [ ]:
movies['keywords']=movies['keywords'].apply(convert)

In [ ]:
movies.head(1)

In [ ]:
movies['cast']

In [ ]:
def find(obj):
    P=[]
    counter=0
    for i in ast.literal_eval(obj):
        if counter !=3:
            P.append(i['name'])
            counter+=1
        else:
            break
    return P


In [ ]:
movies['cast']=movies['cast'].apply(find)

In [ ]:
def fetch_dir(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L
        

In [ ]:
movies['crew']=movies['crew'].apply(fetch_dir)

In [ ]:
movies.head(1)

In [ ]:
movies['overview']=movies['overview'].apply(lambda x:x.split())

In [ ]:
movies['genres']=movies['genres'].apply(lambda x:[i.replace(" ","")for i in x])
movies['keywords']=movies['keywords'].apply(lambda x:[i.replace(" ","")for i in x])
movies['cast']=movies['cast'].apply(lambda x:[i.replace(" ","")for i in x])
movies['crew']=movies['crew'].apply(lambda x:[i.replace(" ","")for i in x])

In [ ]:
movies['tags']=movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']

In [ ]:
!pip install nltk


In [ ]:
import nltk

In [ ]:
from nltk.stem.porter import PorterStemmer
ps= PorterStemmer()

In [ ]:
def stem(text):
    y=[]

    for i in text.split():
        y.append(ps.stem(i))
    return" ".join(y)

In [ ]:
new_df['tags']=new_df['tags'].apply(stem)

In [ ]:
new_df=movies[['id','title','tags']]

In [ ]:
new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x))

In [ ]:
new_df['tags']=new_df['tags'].apply(lambda x:x.lower())

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')

In [ ]:
vectors=cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vectors[1]

In [ ]:
cv.get_feature_names_out()

In [ ]:
similarity=cosine_similarity(vectors)

In [ ]:
similarity[1]

In [ ]:
def recommend(movie):
    movie_index=new_df[new_df['title']==movie].index[0]
    distances=similarity[movie_index]
    movies_list=sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

    
    

In [ ]:
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])[1:6]

In [ ]:
recommend('Batman Begins')

In [ ]:
new_df.iloc[1214].title

In [ ]:
import pickle

In [ ]:
pickle.dump(new_df,open('movie_list.pkl','wb'))

In [ ]:
new_df.iloc[58]['title']

In [ ]:
pickle.dump(similarity,open('similarity.pkl','wb'))